# Old English — Colab 256-byte pilot

Five exploratory arms; 10,000 training examples per arm, fixed seed; 1 epoch each. Run one platform/limit at a time.
This is a pilot, not a full-data replication. Keep completed results off the hosted runtime.


In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile, shutil, json, time
import torch
SESSION_START=time.time()
print('Platform: Google Colab; Python:',sys.version.split()[0], 'Torch:',torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('STOP: Colab has no CUDA GPU. Select Runtime > Change runtime type > T4 GPU. If usage limits block access, wait or use Kaggle; do not run training on CPU.')
print('GPU:',torch.cuda.get_device_name(0))
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout[:1500])
root=Path('/content/oe_project')
required=['src/encoder_experiment.py','data/lex_wikipron.csv','data/brunetti-length.txt','results_wikipron/soliloquies_predictions.jsonl','results_wikipron/tauber_beowulf_predictions.jsonl']
if not all((root/f).is_file() for f in required):
    archives=list(Path('/content').glob('oe_project_final_v3_1*.zip'))
    if not archives:
        from google.colab import files
        print('Upload the original project ZIP (not the notebook package).')
        uploaded=files.upload()
        archives=[Path('/content')/name for name in uploaded if name.startswith('oe_project_final_v3_1') and name.endswith('.zip')]
    assert len(archives)==1, f'STOP: expected exactly one project ZIP, found {archives}'
    with zipfile.ZipFile(archives[0]) as z:
        assert 'oe_project/src/encoder_experiment.py' in z.namelist(), 'Unexpected ZIP layout'
        z.extractall('/content')
for f in required: assert (root/f).is_file(), f'STOP: missing {f}'
print('PROJECT READY:',root)


In [ ]:
import importlib.metadata as md
required_packages={'transformers':'4.57.6','sentencepiece':None,'accelerate':None}
missing=[]
for package,version in required_packages.items():
    try:
        current=md.version(package)
        if version and current!=version: missing.append(f'{package}=={version}')
    except md.PackageNotFoundError:
        missing.append(f'{package}=={version}' if version else package)
if missing:
    print('Installing:',missing,flush=True)
    result=subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',*missing],capture_output=True,text=True)
    if result.returncode:
        print(result.stdout[-2500:]);print(result.stderr[-5000:]);raise RuntimeError('Required package installation failed')
    print('Package installation finished. Warnings about unrelated Gradio/Diffusers do not imply training failure.')
import transformers, accelerate, sentencepiece
from transformers import AutoTokenizer, T5ForConditionalGeneration
assert transformers.__version__=='4.57.6', transformers.__version__
print('Transformers:',transformers.__version__,'Accelerate:',accelerate.__version__)
print('Loading model weights (Internet required on first use)...',flush=True)
tok=AutoTokenizer.from_pretrained('google/byt5-small')
model=T5ForConditionalGeneration.from_pretrained('google/byt5-small')
print('MODEL READY:',round(sum(p.numel() for p in model.parameters())/1e6,1),'million parameters')
del model,tok


## Preflight (short, no real model training)

This runs the existing tiny-model test and confirms expected outputs. It may take several minutes. **Do not continue if it fails.**

In [ ]:
# Tiny smoke test checks actual training + prediction output with a tiny model.
# It is not a full ByT5 GPU memory test.
test_out=Path('/content/encoder_smoke_test')
cmd=[sys.executable,'src/encoder_experiment.py','--tiny','--epochs','1','--arms','word','--out',str(test_out)]
print('Starting tiny smoke test...',flush=True)
p=subprocess.run(cmd,cwd=root,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=900)
print(p.stdout[-12000:]);assert p.returncode==0,f'Smoke test failed ({p.returncode})'
for f in ['metrics.json','soliloquies_predictions.jsonl','tauber_beowulf_predictions.jsonl']:
    assert (test_out/f).is_file(),f'Missing smoke output {f}'
print('SMOKE TEST PASS')

## Lightweight five-arm experiment

Run the wrapper cell, then the runner. All arms use 128 bytes, 1 epoch, batch 32, micro 8, learning rate 0.0003, seed 1. A completed arm is saved and skipped on reruns in the same session. For another Kaggle session, attach the previously downloaded `encoder_light_results.zip` as an input dataset before running. No mid-arm checkpointing exists.

In [ ]:
# Context window implementation is versioned in src/encoder_context_windows.py
# This cell is intentionally a no-op; do not dynamically patch repository code.
print('Context-window script available:', (root/'src/encoder_context_windows.py').exists())
assert (root/'src/encoder_context_windows.py').exists(), 'Update the project dataset from the release package'


In [ ]:
import json, os, subprocess, sys, time, zipfile
from pathlib import Path
os.environ.update(OPENBLAS_NUM_THREADS='1', OMP_NUM_THREADS='1', MKL_NUM_THREADS='1', TOKENIZERS_PARALLELISM='false')
# Exploratory study: fixed 128-byte cap, one epoch, identical optimization settings.
experiments=[('P1','sentence',256,1),('P2','word',256,1),('P3','left',256,1),('P4','local2',256,1),('P5','local5',256,1)]
base=Path('/content/encoder_pilot_256_results');base.mkdir(parents=True,exist_ok=True)
archive=Path('/content/encoder_pilot_256_results.zip')

def pack():
    temp=archive.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temp,'w',zipfile.ZIP_DEFLATED,compresslevel=4) as z:
        for f in sorted(base.rglob('*')):
            if f.is_file():z.write(f,f.relative_to(base.parent))
    temp.replace(archive)
    print('ARCHIVE READY:',archive,'MB:',round(archive.stat().st_size/1e6,2),flush=True)

def validate(eid,arm):
    out=base/eid
    d=json.loads((out/'metrics.json').read_text())
    assert d['epochs']==1 and d['max_bytes']==256 and d.get('train_examples')==10000 and arm in d['arms'], 'Experiment settings mismatch'
    for test in ('soliloquies','tauber_beowulf'):
        seen=set()
        for line in (out/f'{test}_predictions.jsonl').open():
            x=json.loads(line); assert x['id'] not in seen and 'encoder_'+arm in x
            seen.add(x['id'])
        baseline=root/'results_wikipron'/f'{test}_predictions.jsonl'
        original_ids={json.loads(line)['id'] for line in baseline.open()}
        assert seen==original_ids, f'{eid} {test} ID mismatch'
        assert d['arms'][arm][test]['accuracy']['n']==len(seen)
    return True

# Import a previously downloaded ZIP if attached as Kaggle Input. Optional; no prior ZIP is required.
previous=list(Path('/content').glob('encoder_pilot_256_results.zip'))
if len(previous)==1 and not any(base.iterdir()):
    with zipfile.ZipFile(previous[0]) as z:
        for member in z.namelist():
            if member.startswith('encoder_pilot_256_results/') and '..' not in Path(member).parts:
                z.extract(member,'/content')
    print('Restored previous results archive',previous[0])

# The 12-hour Kaggle cap is not guaranteed; stop before a new run if less than 2 hours remain.
# Session age includes all setup time, so define it at the beginning of the notebook when possible.
if 'SESSION_START' not in globals():
    SESSION_START=time.time()
for eid,arm,max_bytes,epochs in experiments:
    out=base/eid;out.mkdir(exist_ok=True)
    done=out/'COMPLETE.json'
    if done.exists():
        validate(eid,arm)
        print('SKIP completed',eid,flush=True)
        continue
    elapsed=time.time()-SESSION_START
    if elapsed > 9*3600:
        pack()
        print('SAFE STOP: notebook has been running over 9h. Download archive before disconnecting.')
        break
    cmd=[sys.executable,'src/encoder_context_windows.py','--train-limit','10000','--arms',arm,'--epochs',str(epochs),'--max-bytes',str(max_bytes),'--batch','32','--micro','8','--lr','0.0003','--seed','1','--out',str(out)]
    print('START',eid,arm,max_bytes,epochs,flush=True)
    start=time.time()
    with (out/'train.log').open('w',encoding='utf8') as f:
        proc=subprocess.Popen(cmd,cwd=root,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=os.environ.copy())
        for line in proc.stdout:
            print(line,end='',flush=True);f.write(line);f.flush()
        status=proc.wait()
    if status!=0:
        pack();raise RuntimeError(f'{eid} failed ({status}); download archive; see train.log')
    validate(eid,arm)
    done.write_text(json.dumps({'id':eid,'arm':arm,'max_bytes':max_bytes,'train_limit':10000,'epochs':epochs,'seed':1,'duration_seconds':round(time.time()-start),'status':'completed'},indent=2))
    pack()
    print('COMPLETE',eid,'minutes',round((time.time()-start)/60,1),flush=True)
else:
    print('ALL FIVE 256-BYTE PILOT EXPERIMENTS COMPLETED')


## Summary / completed results

In [ ]:
import json, zipfile
from pathlib import Path
base=Path('/content/encoder_pilot_256_results')
for eid,arm,_,_ in experiments:
    f=base/eid/'COMPLETE.json'
    if not f.exists():
        print(eid, 'NOT COMPLETED')
        continue
    m=json.loads((base/eid/'metrics.json').read_text())['arms'][arm]
    print(eid,arm,'Soliloquies:',m['soliloquies']['accuracy']['accuracy'],'Beowulf:',m['tauber_beowulf']['accuracy']['accuracy'])
print('Download /content/encoder_pilot_256_results.zip from Output / Files')


## Download results
Run after any completed arm. The archive only survives a reset if you download or persist it.

In [ ]:
from pathlib import Path
p=Path('/content/encoder_light_results.zip')
assert p.is_file(), 'No completed-arm archive yet'
from google.colab import files
files.download(str(p))